In [1]:
# ===========================================================================
# UA-SPEECH MODEL TRAINING - interactive driver
#
# All training logic lives in src/training/; this notebook only calls it and
# stores results, matching notebooks/01_data_pipeline.ipynb's convention -
# functions and architecture belong in src/, only the act of running training
# and storing models happens here.
#
#   src/training/models.py     model factory: acoustic / deep_frozen / deep_lora / fusion
#   src/training/runner.py     TrainingConfig, run_training() - the fold loop
#   src/training/baseline.py   Phase 2: frozen wav2vec + linear SVM baseline
#   src/training/engine.py     one epoch: AMP, gradient clipping, optimizer
#   src/training/metrics.py    accuracy / precision / recall / specificity / F1 / AUROC
#   src/training/reporting.py  predictions / metrics / confusion-matrix / ROC / embeddings I/O
#
# See ROADMAP.md for the phase plan this notebook implements (Phase 1 sanity
# check, Phase 2 baseline reproduction and comparison).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("UA-Speech Training Notebook")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))
print_kv("Speakers", df_m6["Speaker_ID"].nunique())


══════════════════════════════════════════════════════════════════════════════
  UA-SPEECH TRAINING NOTEBOOK
══════════════════════════════════════════════════════════════════════════════
  Manifest ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv
  Utterances .............................. 21381
  Speakers ................................ 28


In [2]:
# STAGE 1 - Pipeline sanity check ("smoke test"). Trains the cheapest model
# (MFCC-only) for one fold, one epoch, on a tiny slice of data. This is NOT a
# real result - it exists to confirm the whole chain (model init, optimizer,
# scheduler, AMP, gradient clipping, early stopping, checkpointing,
# TensorBoard logging, prediction/metric/confusion-matrix/ROC/embedding
# writers) actually runs end to end before spending GPU time on a real run.
from src.training.runner import TrainingConfig, run_training

smoke_cfg = TrainingConfig(
    task="detection", model="acoustic",
    epochs=1, max_folds=1, limit_samples=24,
    run_name="_smoke_test",
)
run_training(df_m6, smoke_cfg)

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│               Model A — MFCC 1D-CNN (cepstral features only)               │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ................................... acoustic — Model A — MFCC 1D-CNN (cepstral features only)
  Cross-validation protocol ............... Leave-One-Speaker-Out
  Run name ................................ _smoke_test
  Device .................................. cuda
  Epochs / batch size ..................... 1 / 8
  LR (head / wav2vec backbone) ............ 0.001 / 0.0001
  Early stopping patience ................. 5 epochs on validation loss

─── Front end — short-time analysis ──────────────────────────────────────────
  Sam


──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: CF02  │  train 24 / val 6 / test 24
──────────────────────────────────────────────────────────────────────────────

─── Architecture — acoustic ──────────────────────────────────────────────────
    acoustic_pathway ......................     103,552 /     103,552 trainable (100.0%)
    classifier ............................       8,386 /       8,386 trainable (100.0%)
  TOTAL trainable ......................... 111,938 / 111,938 (100.00%)



  epoch 1/1 train                    0%|                                   | 0/3 [00:00<?]

  epoch 1/1 train                   33%|██████████▎                    | 1/3 [00:01<00:02]

...

  held-out test (CF02)              67%|████████████████████▋          | 2/3 [00:00<00:00]

  held-out test (CF02)             100%|███████████████████████████████| 3/3 [00:00<00:00]

  Fold CF02 held-out test ................. accuracy=1.000, precision=0.000, recall=0.000, specificity=1.000, f1=0.000, auroc=nan

══════════════════════════════════════════════════════════════════════════════
  RESULTS — _SMOKE_TEST  (1 FOLD(S), 24 HELD-OUT UTTERANCES)
══════════════════════════════════════════════════════════════════════════════

─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean  std
    test_loss 0.5719  NaN
     accuracy 1.0000  NaN
    precision 0.0000  NaN
       recall 0.0000  NaN
  specificity 1.0000  NaN
           f1 0.0000  NaN
        auroc    NaN  NaN

  • Every LOSO fold holds out ONE speaker, who is entirely one class, so per-fold precision / recall / specificity / AUROC above are
  • degenerate — only 'accuracy' is meaningful per fold. The pooled numbers below are the ones comparable to the base paper.

─── Pooled across all folds (base-paper-style LOSO reporting) ────────────────
  accuracy       ██████

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


(                mean  std
 test_loss    0.57194  NaN
 accuracy     1.00000  NaN
 precision    0.00000  NaN
 recall       0.00000  NaN
 specificity  1.00000  NaN
 f1           0.00000  NaN
 auroc            NaN  NaN,
 {'accuracy': 1.0,
  'precision': 0.0,
  'recall': 0.0,
  'specificity': 1.0,
  'f1': 0.0,
  'auroc': nan})

In [3]:
# STAGE 2 - Phase 2, step 1: reproduce the ICASSP base paper's feature
# extractor. Frozen wav2vec 2.0 (no LoRA, no fine-tuning) -> 768-dim
# embedding per utterance. The embedding is identical across every LOSO
# fold, so it is extracted once here and cached to outputs/embeddings/ -
# only the SVM is refit per fold in Stage 3.
from src.training.baseline import extract_frozen_embeddings

frozen_embeddings = extract_frozen_embeddings(df_m6, batch_size=16)
print(f"Frozen embeddings: {frozen_embeddings.shape}")

  Frozen embeddings ....................... loaded from cache (C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\embeddings\frozen_wav2vec_base.npz)
Frozen embeddings: (21381, 768)


In [4]:
# STAGE 3 - Phase 2, step 2: frozen wav2vec 2.0 -> linear SVM, evaluated
# across the full 28-fold LOSO detection protocol - exactly the base paper's
# pipeline. This is the number every other model in this project has to beat
# to be a genuine improvement, not an assumed one.
from src.training.baseline import run_svm_baseline

baseline_summary, baseline_pooled = run_svm_baseline(
    df_m6, task="detection", embeddings=frozen_embeddings, max_folds=None)

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Detection")
print_kv("Accuracy", f"{baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{baseline_pooled['f1']:.4f}")
print_kv("Recall (sensitivity)", f"{baseline_pooled['recall']:.4f}")
print_kv("Precision", f"{baseline_pooled['precision']:.4f}")
print_kv("Specificity", f"{baseline_pooled['specificity']:.4f}")
print_kv("AUROC", f"{baseline_pooled['auroc']:.4f}")


══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run name ................................ baseline_svm_detection


  Fitting SVM per LOSO fold          0%|                                  | 0/28 [00:00<?]

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
  Fitting SVM per LOSO fold          4%|█                             | 1/28 [00:14<06:41]

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
  Fitting SVM per LOSO fold          7%|██▏                           | 2/28 [00:29<06:29]

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
  Fitting SVM per LOSO fold         11%|███▏                          | 3/28 [00:43<06:01]

...

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
  Fitting SVM per LOSO fold         93%|██████████████████████████▉  | 26/28 [06:28<00:32]

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
  Fitting SVM per LOSO fold         96%|███████████████████████████▉ | 27/28 [06:47<00:16]

C:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
  Fitting SVM per LOSO fold        100%|█████████████████████████████| 28/28 [07:07<00:00]

  Fitting SVM per LOSO fold        100%|█████████████████████████████| 28/28 [07:07<00:00]


─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.8021 0.1994
    precision 0.5357 0.5079
       recall 0.4134 0.4358
  specificity 0.3887 0.4280
           f1 0.4504 0.4558
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ████████████████░░░░   0.8020 →
  precision      █████████████████░░░   0.8450  
  recall         ███████████████░░░░░   0.7712  
  specificity    █████████████████░░░   0.8373  
  f1             ████████████████░░░░   0.8064 →
  auroc          █████████████████░░░   0.8649  

══════════════════════════════════════════════════════════════════════════════
  BASELINE (FROZEN WAV2VEC 2.0 + LINEAR SVM) - DETECTION
══════════════════════════════════════════════════════════════════════════════
  Accuracy ................................ 0.8020
  F1 ...................................... 0.8064
  Recall (sensitivity) .........

In [ ]:
# STAGE 4 - Phase 2 step 3 + Phase 3 ablation: full-scale training of all six
# variants - Frozen wav2vec + MLP, LoRA wav2vec + MLP, MFCC CNN, concatenated
# Fusion, and the two Phase 6 attention-fusion variants - on the complete
# 28-fold LOSO detection protocol, full epochs, no sample caps. This is the
# comparison the paper's contribution rests on: fusion -> attention_fusion ->
# attention_fusion_praat, same two pathways/data, only the fusion mechanism
# changes.
#
# NOTE ON SCALE: this is the real run, not a demo - a full 28-fold LOSO pass
# of a wav2vec-fine-tuning variant is hours of GPU time, not minutes, so all
# six variants is realistically a long unattended job. run_training() skips
# any fold whose output already exists on disk and isolates a failing fold
# instead of aborting the whole run (see src/training/runner.py), so
# interrupting this cell and re-running it resumes rather than restarting.
from src.training.runner import TrainingConfig, run_training
from src.training.models import MODEL_NAMES

comparison_pooled = {"baseline_svm": baseline_pooled}

for model_name in MODEL_NAMES:
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg)
    comparison_pooled[model_name] = pooled

In [ ]:
# STAGE 5 - Phase 2/3 comparison table: baseline SVM vs. every trained variant,
# pooled metrics side by side. Saved to outputs/metrics/phase2_comparison.csv
# for the paper/report.
comparison_df = pd.DataFrame(comparison_pooled).T
comparison_df.index.name = "model"

comparison_path = config.METRICS_DIR / "phase2_comparison.csv"
comparison_df.to_csv(comparison_path)

print_header("Phase 2/3 Comparison - Detection")
print_kv("Saved to", comparison_path)
comparison_df

In [ ]:
# STAGE 6 - Severity task: the same six variants on the balanced
# leave-one-speaker-per-class-out protocol (81 folds; see src/splits.py and
# config.DROPPED_FOR_BALANCE). Lower priority than Stage 4's detection run
# (the paper's primary comparison) - run this once Stage 4 is done or far
# enough along, since it is a larger job (81 folds vs. 28) on the same GPU.
from src.training.runner import TrainingConfig, run_training
from src.training.models import MODEL_NAMES

severity_pooled = {}

for model_name in MODEL_NAMES:
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg)
    severity_pooled[model_name] = pooled

severity_df = pd.DataFrame(severity_pooled).T
severity_df.index.name = "model"

severity_path = config.METRICS_DIR / "phase3_severity_comparison.csv"
severity_df.to_csv(severity_path)

print_header("Phase 3 Comparison - Severity")
print_kv("Saved to", severity_path)
severity_df